# 54. 核密度图（kdeplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 11 / 20 步：读懂连续变量的整体分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 直方图（histplot）  →  **本章任务：** 核密度图（kdeplot）  →  **下一步：** 累积分布图（ecdfplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

做数据分析时，仅靠平均数或最大值往往看不出整体分布长什么样——数据是集中在一个值附近，还是拉出一条又长又细的尾巴？核密度图（kdeplot）把一串数值的分布画成一条连续平滑的曲线，让"哪里更密集、哪里更稀疏"一目了然，特别适合在样本量较充足时比较不同分组或不同时间段的整体形状，比直方图更容易看清分布的细节。



## 本章目标

学完本章，你将能够：

- **理解**：理解「核密度图（kdeplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「核密度图（kdeplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「核密度图（kdeplot）」并读出其中的结论。


## 54.1 适用场景

**背景引入**：做数据分析时，仅靠平均数或最大值往往看不出整体分布长什么样——数据是集中在一个值附近，还是拉出一条又长又细的尾巴？核密度图（kdeplot）把一串数值的分布画成一条连续平滑的曲线，让"哪里更密集、哪里更稀疏"一目了然，特别适合在样本量较充足时比较不同分组或不同时间段的整体形状，比直方图更容易看清分布的细节。掌握它后，你就能用一张图讲清楚"数据到底集中在哪里、分布长相如何"。

样本量较充足，希望比较连续分布的整体形状。


## 54.2 数据结构

连续数值样本；分组KDE要求每组有足够且不完全相同的数值。


## 54.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 bw_adjust 参数从 0.9 改为 0.5 或 1.5，观察带宽对密度曲线平滑度的影响
2. 修改 common_norm=False 为 common_norm=True，对比独立归一化与共同归一化的曲线高度
3. 调整 cut 参数从 0 为 3，说明边界延伸对密度估计范围的影响


## 54.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.kdeplot()`、`ax.set()`、`fig.tight_layout()` | 样本量较充足，希望比较连续分布的整体形状。 | 小样本使用KDE |
| 进阶变体 | `plt.subplots()`、`sns.kdeplot()`、`ax.set()`、`fig.tight_layout()` | 在基础图表上增加分组、注释、布局或交互 | 边界外出现不可能值 |
| 关键参数 | `bw_adjust` | 带宽 | 小样本使用KDE |
| 关键参数 | `fill` | 填充 | 边界外出现不可能值 |
| 关键参数 | `common_norm` | 共同归一化 | 把平滑峰当作真实分组 |
| 关键参数 | `cut` | 边界延伸 | 小样本使用KDE |


## 54.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-54 -->
### 数学推导｜核密度估计把样本平滑成分布

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜每个样本放置一个核。** 以 $x_i$ 为中心、带宽为 $h$ 的核为

$$
K_h(x-x_i)=\frac{1}{h}K\!\left(\frac{x-x_i}{h}\right)
$$

$1/h$ 保证拉宽曲线后面积仍为 1。

**第 2 步｜把所有小曲线平均。** $n$ 个单位面积核相加后再除以 $n$，总面积仍为 1，于是得到 $\hat f_h(x)$。

**第 3 步｜理解带宽。** 较小 $h$ 保留局部起伏但方差大；较大 $h$ 更平滑但可能抹掉真实结构。

**把上面的关系收束为本章计算式：**

$$
\hat{f}_h(x)=\frac{1}{nh}\sum_{i=1}^{n}K\!\left(\frac{x-x_i}{h}\right)
$$

**符号解释：** $K$ 是核函数，$h$ 是带宽；$h$ 越大曲线越平滑。

**代码对应：** 调整 `bw_adjust`（或旧版带宽参数）并与原始样本/直方图交叉检查。

**使用边界：** KDE 会在观测范围外延伸，小样本或有自然边界的数据尤其要谨慎。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 54.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import scipy

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.3))
sns.kdeplot(
    data=orders, x="order_value", fill=True, color="#1a73e8", cut=0, ax=ax
)
ax.set(title="订单金额核密度", xlabel="客单价（元）", ylabel="密度")
fig.tight_layout()
plt.show()


**练一练**：在第 48.4 节基础图表里，订单金额核密度只画了一条曲线。现在请只改动一个数据字段：把 `x="order_value"`（客单价）换成 `x="items"`（克拉数），再跑一次，观察两者分布形状的差别（比如 `order_value` 更偏向高值拖长尾，还是 `items` 更偏向低值聚集）。改完后用一句话写下你观察到的差异，并回到答案自检核对自己的写法。


In [ ]:
# 请在下方填写代码
# 任务：只改一个数据字段，把基础图表里的 x 从 "order_value" 换成 "items"，
#       观察密度曲线形状的变化。填空后运行即可。

fig, ax = plt.subplots(figsize=(8, 4.3))
sns.kdeplot(
    data=orders,
    x=____,  # 请把 ____ 填成要观察的列名，如 "items"
    fill=True,
    cut=0,
    ax=ax,
)


In [ ]:
# ===== 完整答案：把 x 换成 "items"（克拉数），观察其偏向低值的分布 =====
fig, ax = plt.subplots(figsize=(8, 4.3))
sns.kdeplot(
    data=orders,
    x="items",
    fill=True,
    color="#D93025",
    cut=0,
    ax=ax,
)
ax.set(title="克拉数核密度", xlabel="克拉", ylabel="密度")
fig.tight_layout()


## 54.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.kdeplot(
    data=orders,
    x="order_value",
    hue="category",
    common_norm=False,
    bw_adjust=0.9,
    cut=0,
    linewidth=2,
    palette="colorblind",
    ax=ax,
)
ax.set(title="品类客单价密度", xlabel="客单价（元）", ylabel="密度")
fig.tight_layout()
plt.show()


## 54.8 参数说明

- bw_adjust：带宽
- fill：填充
- common_norm：共同归一化
- cut：边界延伸


## 54.9 结果解读

曲线面积表示概率密度，峰高不是样本数；用不同带宽检查峰形稳定性。


## 54.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 54.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 54.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 54.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 54.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 54.12 易错点提醒

- 小样本使用KDE
- 边界外出现不可能值
- 把平滑峰当作真实分组


## 54.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 54.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：按渠道分层密度，比较不同渠道的分布形状
# 【目标】分层核密度，看不同渠道的分布形状(如峰、尾巴)有何不同。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 hue="channel"，用半透明色叠加各渠道的密度曲线。
fig, ax = plt.subplots(figsize=(8, 4.3))
sns.kdeplot(
    data=orders,
    x="order_value",
    hue="channel",
    fill=True,
    alpha=0.35,
    cut=0,
    ax=ax,
)
ax.set(title="分渠道订单金额核密度", xlabel="客单价（元）", ylabel="密度")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：各渠道密度曲线的山峰位置与宽度有何差异 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.kdeplot(
    data=marketing, x="sales", bw_adjust=0.45, color="#188038", ax=axes[0]
)
axes[0].set(title="较小带宽")
sns.kdeplot(
    data=marketing, x="sales", bw_adjust=1.6, color="#188038", ax=axes[1]
)
axes[1].set(title="较大带宽")
fig.tight_layout()
plt.show()


## 54.15 小结

用核密度估计平滑展示分布，并理解带宽和边界的影响。


### 54.15.1 你已经掌握

- 判断核密度图（kdeplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 54.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `bw_adjust` | 带宽 |
| `fill` | 填充 |
| `common_norm` | 共同归一化 |
| `cut` | 边界延伸 |


### 54.15.3 需要注意

- 小样本使用KDE
- 边界外出现不可能值
- 把平滑峰当作真实分组


### 54.15.4 完成检查

- [ ] 能判断什么问题适合使用核密度图（kdeplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 54.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
